## This notebook uses the result to generate figs and tables

## define functions for metrics calculations

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score
)


def compute_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return {
        'threshold': threshold, 'sensitivity': sensitivity,
        'specificity': specificity, 'ppv': ppv, 'npv': npv, 'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    }


# --- Threshold selection (on TRAIN data) ---

def find_threshold_high_sensitivity(y_true, y_prob, min_sensitivity=0.90):
    thresholds = np.linspace(0.01, 0.99, 200)
    best = None
    for t in thresholds:
        m = compute_metrics_at_threshold(y_true, y_prob, t)
        if m['sensitivity'] >= min_sensitivity and (best is None or t > best['threshold']):
            best = m
    return best['threshold'] if best else None


def find_threshold_high_specificity(y_true, y_prob, min_specificity=0.90):
    thresholds = np.linspace(0.01, 0.99, 200)
    best = None
    for t in thresholds:
        m = compute_metrics_at_threshold(y_true, y_prob, t)
        if m['specificity'] >= min_specificity and (best is None or t < best['threshold']):
            best = m
    return best['threshold'] if best else None


def find_threshold_balanced(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 200)
    best, best_diff = None, np.inf
    for t in thresholds:
        m = compute_metrics_at_threshold(y_true, y_prob, t)
        diff = abs(m['sensitivity'] - m['specificity'])
        if diff < best_diff:
            best_diff, best = diff, m
    return best['threshold']


def find_threshold_youdens(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 200)
    best, best_j = None, -np.inf
    for t in thresholds:
        m = compute_metrics_at_threshold(y_true, y_prob, t)
        j = m['sensitivity'] + m['specificity'] - 1
        if j > best_j:
            best_j, best = j, m
    return best['threshold']


def calibration_slope_intercept(y_true, y_prob):
    eps = 1e-6
    # Calculate logit of probabilities
    logit_prob = logit(np.clip(y_prob, eps, 1 - eps)).reshape(-1, 1)
    
    # Must use LogisticRegression, not LinearRegression
    lr = LogisticRegression(penalty=None)
    lr.fit(logit_prob, y_true)
    
    return lr.coef_[0][0], lr.intercept_[0]


def evaluate_model(y_train, y_prob_train, y_test, y_prob_test, model_name="Model"):
    """
    Thresholds are selected on (y_train, y_prob_train),
    then applied to (y_test, y_prob_test) for final evaluation.
    Global metrics (AUC, AUPRC, Brier, calibration) are computed on test set.
    """
    y_train, y_prob_train = np.array(y_train), np.array(y_prob_train)
    y_test,  y_prob_test  = np.array(y_test),  np.array(y_prob_test)

    roc_auc  = roc_auc_score(y_test, y_prob_test)
    auprc    = average_precision_score(y_test, y_prob_test)
    brier    = brier_score_loss(y_test, y_prob_test)
    slope, intercept = calibration_slope_intercept(y_test, y_prob_test)

    # Thresholds from TRAIN, metrics evaluated on TEST
    strategies = [
        ("High Sensitivity (≥0.90)", "high_sensitivity",find_threshold_high_sensitivity(y_train, y_prob_train)),
        ("Balanced (Sens ≈ Spec)",   "balanced",
            find_threshold_balanced(y_train, y_prob_train)),
        ("High Specificity (≥0.90)", "high_specificity",
            find_threshold_high_specificity(y_train, y_prob_train)),
        ("Youden's Index",           "youdens_index",
            find_threshold_youdens(y_train, y_prob_train)),
    ]

    print(f"\n{'='*55}")
    print(f"  {model_name} — Evaluation Report (test set)")
    print(f"{'='*55}")
    print(f"  ROC-AUC:              {roc_auc:.4f}")
    print(f"  AUPRC:                {auprc:.4f}")
    print(f"  Brier Score:          {brier:.4f}")
    print(f"  Calibration Slope:    {slope:.4f}  (ideal: 1.0)")
    print(f"  Calibration Intercept:{intercept:.4f}  (ideal: 0.0)")

    rows = []
    for label, key, threshold in strategies:
        if threshold is None:
            print(f"\n  [{label}] — threshold not achievable on train set")
            continue
        m = compute_metrics_at_threshold(y_test, y_prob_test, threshold)
        j = m['sensitivity'] + m['specificity'] - 1
        print(f"\n  [{label}]  (threshold selected on train)")
        print(f"    Threshold:   {threshold:.3f}")
        print(f"    Sensitivity: {m['sensitivity']:.4f}")
        print(f"    Specificity: {m['specificity']:.4f}")
        print(f"    PPV:         {m['ppv']:.4f}")
        print(f"    NPV:         {m['npv']:.4f}")
        print(f"    F1:          {m['f1']:.4f}")
        print(f"    Youden's J:  {j:.4f}")
        print(f"    TP={m['tp']}  TN={m['tn']}  FP={m['fp']}  FN={m['fn']}")
        rows.append({
            'threshold_strategy': key, **m, 'youdens_j': j,
            'roc_auc': roc_auc, 'auprc': auprc, 'brier_score': brier,
            'calibration_slope': slope, 'calibration_intercept': intercept
        })

    print(f"{'='*55}\n")
    return pd.DataFrame(rows)


# --- Usage ---
# summary_df = evaluate_model(y_train, y_prob_train, y_test, y_prob_test, model_name="MLP")
# summary_df.to_csv("evaluation_metrics.csv", index=False)


## calculating metrics for each model across various thresholds

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# ==================================================
# MODELS
# ==================================================

models = [
    "svm",
    "xgb",
    "bbn",
    "nn",
    "lr",
    "rf",
    "stack"
]

n_folds = 5
n_imputations = 5

# ==================================================
# STORAGE
# ==================================================

all_train_preds = []
all_test_preds = []
all_fold_metrics = []

# ==================================================
# LOOP OVER FOLDS
# ==================================================

for fold in range(1, n_folds + 1):

    # ---------------------------
    # TRUE LABELS
    # ---------------------------
    y_train = pd.read_csv(
        f"Fold_{fold}_predictions_svm_train.csv"
    )['y_true'].values.squeeze()

    y_test = pd.read_csv(
        f"Fold_{fold}_predictions_svm_test.csv"
    )['y_true'].values.squeeze()

    # ==================================================
    # ML MODELS
    # ==================================================
    for model in models:

        y_pred_train = pd.read_csv(
            f"Fold_{fold}_predictions_{model}_train.csv"
        ).iloc[:, 1].values.squeeze()

        y_pred_test = pd.read_csv(
            f"Fold_{fold}_predictions_{model}_test.csv"
        ).iloc[:, 1].values.squeeze()

        # ---------------------------
        # STORE RAW PREDICTIONS
        # ---------------------------
        all_train_preds.append(pd.DataFrame({
            "Fold": fold,
            "Model": model,
            "y_true": y_train,
            "y_pred": y_pred_train
        }))

        all_test_preds.append(pd.DataFrame({
            "Fold": fold,
            "Model": model,
            "y_true": y_test,
            "y_pred": y_pred_test
        }))

        # ---------------------------
        # EVALUATE
        # ---------------------------
        metrics_df = evaluate_model(
            y_train, y_pred_train,
            y_test, y_pred_test,
            model_name=model
        )

        metrics_df["Fold"] = fold
        metrics_df["Model"] = model
        all_fold_metrics.append(metrics_df)

    # ==================================================
    # CLINICAL SCORES (SOFA & APACHE)
    # ==================================================

    sofa_test_imputs = []
    apache_test_imputs = []
    sofa_train_imputs = []
    apache_train_imputs = []

    for m in range(1, n_imputations + 1):

        df_train = pd.read_spss(
            f"Fold_{fold}/Train_Imputed_m{m}.sav"
        )

        df_test = pd.read_spss(
            f"Fold_{fold}/Test_Imputed_m{m}.sav"
        )

        sofa_test_imputs.append(
            df_test['Sofa_Mean'].values / df_train['Sofa_Mean'].max()
        )
        apache_test_imputs.append(
            df_test['Apache'].values / df_train['Apache'].max()
        )

        sofa_train_imputs.append(
            df_train['Sofa_Mean'].values / df_train['Sofa_Mean'].max()
        )
        apache_train_imputs.append(
            df_train['Apache'].values / df_train['Apache'].max()
        )

    # ---------------------------
    # RUBIN'S RULES (MEAN IMPUTATIONS)
    # ---------------------------
    y_pred_train_sofa = np.mean(sofa_train_imputs, axis=0)
    y_pred_test_sofa = np.mean(sofa_test_imputs, axis=0)

    y_pred_train_apache = np.mean(apache_train_imputs, axis=0)
    y_pred_test_apache = np.mean(apache_test_imputs, axis=0)

    # ==================================================
    # PLATT CALIBRATION FUNCTION
    # ==================================================

    def platt_calibration(x_train, y_train, x_test):
        """
        Platt scaling using calibrated logistic regression
        """
        X_train = x_train.reshape(-1, 1)
        X_test = x_test.reshape(-1, 1)

        base_lr = LogisticRegression(solver="lbfgs")

        calibrator = CalibratedClassifierCV(
            estimator=base_lr,
            method="sigmoid",
            cv=3
        )

        calibrator.fit(X_train, y_train)

        return (
            calibrator.predict_proba(X_train)[:, 1],
            calibrator.predict_proba(X_test)[:, 1]
        )

    # ==================================================
    # APPLY TO CLINICAL SCORES
    # ==================================================

    for clinical_model, train_score, test_score in [
        ("SOFA", y_pred_train_sofa, y_pred_test_sofa),
        ("APACHE", y_pred_train_apache, y_pred_test_apache)
    ]:

        y_train_prob, y_test_prob = platt_calibration(
            train_score,
            y_train,
            test_score
        )

        # ---------------------------
        # STORE TRAIN
        # ---------------------------
        all_train_preds.append(pd.DataFrame({
            "Fold": fold,
            "Model": clinical_model,
            "y_true": y_train,
            "y_pred": y_train_prob
        }))

        # ---------------------------
        # STORE TEST
        # ---------------------------
        all_test_preds.append(pd.DataFrame({
            "Fold": fold,
            "Model": clinical_model,
            "y_true": y_test,
            "y_pred": y_test_prob
        }))

        # ---------------------------
        # EVALUATE
        # ---------------------------
        metrics_df_clin = evaluate_model(
            y_train, y_train_prob,
            y_test, y_test_prob,
            model_name=clinical_model
        )

        metrics_df_clin["Fold"] = fold
        metrics_df_clin["Model"] = clinical_model
        all_fold_metrics.append(metrics_df_clin)

# ==================================================
# FINAL OUTPUTS
# ==================================================

train_predictions_df = pd.concat(all_train_preds, ignore_index=True)
test_predictions_df = pd.concat(all_test_preds, ignore_index=True)
fold_metrics_df = pd.concat(all_fold_metrics, ignore_index=True)

# ==================================================
# SUMMARY STATISTICS
# ==================================================

metric_cols = [
    "sensitivity",
    "specificity",
    "ppv",
    "npv",
    "f1",
    "roc_auc",
    "auprc",
    "brier_score",
    "calibration_slope",
    "calibration_intercept"
]

metrics_summary_df = (
    fold_metrics_df
    .groupby(["Model", "threshold_strategy"])[metric_cols]
    .agg(["mean", "std"])
)

metrics_summary_df.columns = [
    f"{m}_{s}" for m, s in metrics_summary_df.columns
]

metrics_summary_df = metrics_summary_df.reset_index()

# ==================================================
# SAVE FILES
# ==================================================

metrics_summary_df.to_csv("metrics_summary_mean_stdN.csv", index=False)
train_predictions_df.to_csv("all_train_predictionsN.csv", index=False)
test_predictions_df.to_csv("all_test_predictionsN.csv", index=False)


  svm — Evaluation Report (test set)
  ROC-AUC:              0.9195
  AUPRC:                0.8290
  Brier Score:          0.0915
  Calibration Slope:    1.2213  (ideal: 1.0)
  Calibration Intercept:-0.0507  (ideal: 0.0)

  [High Sensitivity (≥0.90)]  (threshold selected on train)
    Threshold:   0.286
    Sensitivity: 0.9111
    Specificity: 0.8116
    PPV:         0.6119
    NPV:         0.9655
    F1:          0.7321
    Youden's J:  0.7227
    TP=41  TN=112  FP=26  FN=4

  [Balanced (Sens ≈ Spec)]  (threshold selected on train)
    Threshold:   0.315
    Sensitivity: 0.8444
    Specificity: 0.8188
    PPV:         0.6032
    NPV:         0.9417
    F1:          0.7037
    Youden's J:  0.6633
    TP=38  TN=113  FP=25  FN=7

  [High Specificity (≥0.90)]  (threshold selected on train)
    Threshold:   0.414
    Sensitivity: 0.7111
    Specificity: 0.8841
    PPV:         0.6667
    NPV:         0.9037
    F1:          0.6882
    Youden's J:  0.5952
    TP=32  TN=122  FP=16  FN=13

 

In [3]:
import pandas as pd
metrics_summary_df = pd.read_csv('metrics_summary_mean_stdN.csv')
display_df = metrics_summary_df.copy()

metrics = [
    "sensitivity", 
    "specificity",
    "ppv",
    "npv",
    "f1",
    "roc_auc",
    "auprc",
    "brier_score",
    "calibration_slope",
    "calibration_intercept"
]

for metric in metrics:
    print(metric)
    display_df[metric] = (
        display_df[f"{metric}_mean"].astype(float).round(3).astype(str)
        + " ± "
        + display_df[f"{metric}_std"].astype(float).round(3).astype(str)
    )

display_df = display_df[
    ["Model", "threshold_strategy"] + metrics
]
display_df.sort_values(by='threshold_strategy').astype(str).to_csv('showinfdf.csv')

sensitivity
specificity
ppv
npv
f1
roc_auc
auprc
brier_score
calibration_slope
calibration_intercept


In [4]:
display_df = display_df.sort_values(by='threshold_strategy') 
replacements = {
    'stack': 'Stacking',
    'xgb': 'XGBoost',
    'nn': 'MLP' 
}

display_df['Model'] = display_df['Model'].replace(replacements)
display_df

,Model,threshold_strategy,sensitivity,specificity,ppv,npv,f1,roc_auc,auprc,brier_score,calibration_slope,calibration_intercept
0,APACHE,balanced,0.756 ± 0.058,0.761 ± 0.015,0.504 ± 0.033,0.907 ± 0.02,0.605 ± 0.041,0.852 ± 0.033,0.635 ± 0.063,0.13 ± 0.013,1.058 ± 0.192,0.036 ± 0.142
32,XGBoost,balanced,0.73 ± 0.064,0.865 ± 0.04,0.641 ± 0.046,0.909 ± 0.017,0.68 ± 0.027,0.912 ± 0.008,0.755 ± 0.022,0.105 ± 0.005,1.143 ± 0.052,0.098 ± 0.222
4,SOFA,balanced,0.815 ± 0.011,0.81 ± 0.037,0.584 ± 0.046,0.932 ± 0.001,0.679 ± 0.027,0.895 ± 0.008,0.733 ± 0.019,0.11 ± 0.004,1.09 ± 0.193,0.087 ± 0.313
28,svm,balanced,0.802 ± 0.039,0.849 ± 0.028,0.634 ± 0.033,0.93 ± 0.011,0.707 ± 0.019,0.915 ± 0.007,0.782 ± 0.031,0.101 ± 0.006,1.181 ± 0.147,0.114 ± 0.218
24,Stacking,balanced,0.797 ± 0.036,0.864 ± 0.017,0.654 ± 0.029,0.93 ± 0.012,0.718 ± 0.028,0.917 ± 0.007,0.777 ± 0.02,0.1 ± 0.006,1.036 ± 0.067,0.025 ± 0.184
8,bbn,balanced,0.869 ± 0.037,0.774 ± 0.03,0.554 ± 0.022,0.949 ± 0.013,0.676 ± 0.014,0.9 ± 0.003,0.686 ± 0.024,0.106 ± 0.002,1.074 ± 0.26,0.026 ± 0.169
20,rf,balanced,0.712 ± 0.08,0.859 ± 0.044,0.626 ± 0.05,0.904 ± 0.021,0.662 ± 0.023,0.908 ± 0.008,0.74 ± 0.022,0.107 ± 0.005,1.1 ± 0.082,0.067 ± 0.219
16,MLP,balanced,0.766 ± 0.076,0.87 ± 0.021,0.655 ± 0.027,0.921 ± 0.023,0.704 ± 0.032,0.915 ± 0.01,0.79 ± 0.035,0.099 ± 0.008,1.098 ± 0.194,0.07 ± 0.186
12,lr,balanced,0.824 ± 0.053,0.842 ± 0.03,0.629 ± 0.036,0.937 ± 0.016,0.712 ± 0.029,0.916 ± 0.008,0.78 ± 0.028,0.1 ± 0.007,1.141 ± 0.113,0.076 ± 0.209
33,XGBoost,high_sensitivity,0.743 ± 0.106,0.872 ± 0.048,0.661 ± 0.049,0.915 ± 0.03,0.693 ± 0.023,0.912 ± 0.008,0.755 ± 0.022,0.105 ± 0.005,1.143 ± 0.052,0.098 ± 0.222


In [5]:
display_df.sort_values(by='threshold_strategy').to_csv('output.csv', index=False, encoding='utf-8-sig')


## plotting

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. JOURNAL-STYLE TYPOGRAPHY & SIZE CONFIG
# ==========================================
plt.style.use("seaborn-v0_8-whitegrid")  
plt.rcParams.update({
    "font.family": "serif",          # Consistent with your ROC/Calibration script
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 14,
    "legend.fontsize": 10,
    "legend.frameon": False,        # Clean minimalist layout
    "axes.edgecolor": "#CCCCCC",
    "savefig.dpi": 600,             # High-resolution print-ready
    "savefig.bbox": "tight"
})

# Complete explicit aesthetic mapping (Identical palette for publication continuity)
MODEL_COLORS = {
    'stack':   '#E6194B',  # Vivid Red
    'xgb':     '#F58231',  # Bright Orange
    'rf':      "#526138",  # Vibrant Green
    'svm':     '#4363D8',  # Strong Blue
    'lr':      '#911EB4',  # High-Contrast Purple
    'nn':      '#42D4F4',  # Bright Cyan/Teal
    'bbn':     '#C51162',  # Vivid Magenta/Plum
    'sofa':    "#20A675",  # Deep Amber/Gold (darker than previous to show up on white)
    'apache':  '#212121'   # Solid Dark Charcoal/Almost Black for sharp contrast
}

MODEL_LABELS = {
    'stack': 'Stacking Ensemble',
    'xgb': 'XGBoost',
    'rf': 'Random Forest',
    'svm': 'SVM',
    'lr': 'Logistic Regression',
    'nn': 'Neural Network',
    'bbn': 'Bayesian Belief Network',
    'SOFA': 'SOFA Score',
    'APACHE': 'APACHE Score'
}

# Standardized list matching your specified cohort models
MODELS_TO_PLOT  = ['stack', 'xgb', 'lr','svm','rf','nn','bbn', 'SOFA', 'APACHE']


# ==========================================
# 2. DECISION CURVE ANALYSIS ENGINE
# ==========================================
def calculate_net_benefit(y_true, y_pred_prob, thresholds):
    """Calculates Net Benefit at specified threshold probabilities."""
    net_benefits = []
    n = len(y_true)

    for t in thresholds:
        if t == 0:
            t = 1e-5
        if t == 1:
            t = 1 - 1e-5

        # Count True Positives and False Positives based on the threshold
        tp = np.sum((y_pred_prob >= t) & (y_true == 1))
        fp = np.sum((y_pred_prob >= t) & (y_true == 0))

        # Net Benefit formula: (TP / N) - (FP / N) * (pt / (1 - pt))
        nb = (tp / n) - (fp / n) * (t / (1 - t))
        net_benefits.append(nb)

    return np.array(net_benefits)


def plot_decision_curve_reviewed(csv_path, thresholds=None):
    print(f"Reading variables from local path: {csv_path}")
    
    # LOW-RAM OPTIMIZATION: Read only required target vectors
    needed_cols = ['Model', 'y_true', 'y_pred']
    dtypes = {'Model': 'category', 'y_true': 'int8', 'y_pred': 'float32'}
    raw_df = pd.read_csv(csv_path, usecols=needed_cols, dtype=dtypes)
    
    # Standardize string structures
    raw_df['Model'] = raw_df['Model'].astype(str).str.lower().str.strip()
    
    # Filter to isolate target models
    target_models = [m.lower().strip() for m in MODELS_TO_PLOT]
    df = raw_df[raw_df['Model'].isin(target_models)].copy()
    
    unique_models = df['Model'].unique()
    print(f"Plotting DCA for models: {list(unique_models)}")

    if thresholds is None:
        # Truncating at 0.85 to eliminate high-threshold unstable noise artifacts
        thresholds = np.linspace(0.01, 0.85, 100)

    fig_size = (7.5, 6.5)
    fig, ax = plt.subplots(figsize=fig_size)

    # 1. Baseline: Treat All (Calculated using clinical prevalence)
    # Extract true outcomes from any valid subset slice to verify baseline footprint
    sample_model = unique_models[0]
    y_all_true = df[df["Model"] == sample_model]["y_true"].values
    prevalence = np.mean(y_all_true)
    
    tp_all = np.sum(y_all_true == 1)
    fp_all = np.sum(y_all_true == 0)
    n_all = len(y_all_true)
    nb_all = (tp_all / n_all) - (fp_all / n_all) * (thresholds / (1 - thresholds))

    ax.plot(thresholds, nb_all, color="darkgray", linestyle="--", label="Treat All", alpha=0.8, linewidth=1.3)

    # 2. Baseline: Treat None (Horizontal zero anchor line)
    ax.plot(thresholds, np.zeros_like(thresholds), color="black", linestyle="-", label="Treat None", linewidth=1.5)

    # 3. Dynamic Model Curve Iterations
    for model_name in unique_models:
        model_df = df[df["Model"] == model_name]
        y_true = model_df["y_true"].values
        y_pred = model_df["y_pred"].values

        # Compute clinical Net Benefit curve array
        nb_model = calculate_net_benefit(y_true, y_pred, thresholds)

        # Apply standardized color and title schemas
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8

        ax.plot(thresholds, nb_model, color=color, linewidth=linewidth, label=label)

    # Academic Formatting Adjustments 
    ax.set_xlim(0, 0.85)
    ax.set_ylim(ymin=-0.02, ymax=max(prevalence * 1.15, 0.25))

    ax.set_xlabel("Threshold Probability ($p_t$)", labelpad=10)
    ax.set_ylabel("Net Benefit", labelpad=10)
    ax.set_title("Decision Curve Analysis", pad=15)

    # CRITICAL UPDATE: Shifted legend completely outside to the right of the grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))

    # Clean borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)

    # Save at 600 DPI; bbox_inches='tight' will auto-expand the canvas for the out-of-bounds legend
    plt.savefig("decision_curve_final_publication.pdf", format='pdf', bbox_inches="tight")
    plt.close(fig)
    print("Saved: decision_curve_final_publication.pdf")


# ==========================================
# 3. RUN EXECUTION
# ==========================================

local_csv_path = 'all_test_predictionsN.csv'
plot_decision_curve_reviewed(local_csv_path)

Reading variables from local path: all_test_predictionsN.csv
Plotting DCA for models: ['svm', 'xgb', 'bbn', 'nn', 'lr', 'rf', 'stack', 'sofa', 'apache']
Saved: decision_curve_final_publication.pdf


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. JOURNAL-STYLE TYPOGRAPHY & SIZE CONFIG
# ==========================================
plt.style.use("seaborn-v0_8-whitegrid")  
plt.rcParams.update({
    "font.family": "serif",          # Consistent with your ROC/Calibration script
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.titlesize": 12,
    "legend.fontsize": 12,
    "legend.frameon": False,        # Clean minimalist layout
    "axes.edgecolor": "#CCCCCC",
    "savefig.dpi": 600,             # High-resolution print-ready
    "savefig.bbox": "tight"
})

# Complete explicit aesthetic mapping (Identical palette for publication continuity)
MODEL_COLORS = {
    'stack':   '#d62728',  # Bold Crimson
    'xgb':     '#ff7f0e',  # Safety Orange
    'rf':      "#2c0c3a",  # Forest Green
    'svm':     '#1f77b4',  # Soft Blue
    'lr':      '#9467bd',  # Royal Purple
    'nn':      '#17becf',  # Deep Teal
    'bbn':     '#800040',  # Deep Plum Burgundy
    'sofa':    "#267c40",  # Amber Gold  ← was duplicate green
    'apache':  '#4a5568'   # Slate Charcoal
}

MODEL_LABELS = {
    'stack': 'STACKING',
    'xgb': 'XGBoost',
    'rf': 'Random Forest',
    'svm': 'SVM',
    'lr': 'Logistic Regression',
    'nn': 'Neural Network',
    'bbn': 'Bayesian Belief Network',
    'SOFA': 'SOFA Score',
    'APACHE': 'APACHE Score'
}

# Standardized list matching your specified cohort models
MODELS_TO_PLOT = ['stack', 'xgb', 'lr', 'rf', 'bbn', 'nn', 'svm', 'SOFA', 'APACHE']

# ==========================================
# 2. DECISION CURVE ANALYSIS ENGINE
# ==========================================
def calculate_net_benefit(y_true, y_pred_prob, thresholds):
    """Calculates Net Benefit at specified threshold probabilities."""
    net_benefits = []
    n = len(y_true)

    for t in thresholds:
        if t == 0:
            t = 1e-5
        if t == 1:
            t = 1 - 1e-5

        # Count True Positives and False Positives based on the threshold
        tp = np.sum((y_pred_prob >= t) & (y_true == 1))
        fp = np.sum((y_pred_prob >= t) & (y_true == 0))

        # Net Benefit formula: (TP / N) - (FP / N) * (pt / (1 - pt))
        nb = (tp / n) - (fp / n) * (t / (1 - t))
        net_benefits.append(nb)

    return np.array(net_benefits)


def plot_decision_curve_reviewed(csv_path, thresholds=None):
    print(f"Reading variables from local path: {csv_path}")
    
    # LOW-RAM OPTIMIZATION: Read only required target vectors
    needed_cols = ['Model', 'y_true', 'y_pred']
    dtypes = {'Model': 'category', 'y_true': 'int8', 'y_pred': 'float32'}
    raw_df = pd.read_csv(csv_path, usecols=needed_cols, dtype=dtypes)
    
    # Standardize string structures
    raw_df['Model'] = raw_df['Model'].astype(str).str.lower().str.strip()
    
    # Filter to isolate target models
    target_models = [m.lower().strip() for m in MODELS_TO_PLOT]
    df = raw_df[raw_df['Model'].isin(target_models)].copy()
    
    unique_models = df['Model'].unique()
    print(f"Plotting DCA for models: {list(unique_models)}")

    if thresholds is None:
        # Truncating at 0.85 to eliminate high-threshold unstable noise artifacts
        thresholds = np.linspace(0.01, 0.85, 100)

    fig_size = (7.5, 6.5)
    fig, ax = plt.subplots(figsize=fig_size)

    # 1. Baseline: Treat All (Calculated using clinical prevalence)
    # Extract true outcomes from any valid subset slice to verify baseline footprint
    sample_model = unique_models[0]
    y_all_true = df[df["Model"] == sample_model]["y_true"].values
    prevalence = np.mean(y_all_true)
    
    tp_all = np.sum(y_all_true == 1)
    fp_all = np.sum(y_all_true == 0)
    n_all = len(y_all_true)
    nb_all = (tp_all / n_all) - (fp_all / n_all) * (thresholds / (1 - thresholds))

    ax.plot(thresholds, nb_all, color="darkgray", linestyle="--", label="Treat All", alpha=0.8, linewidth=1.3)

    # 2. Baseline: Treat None (Horizontal zero anchor line)
    ax.plot(thresholds, np.zeros_like(thresholds), color="black", linestyle="-", label="Treat None", linewidth=1.5)

    # 3. Dynamic Model Curve Iterations
    for model_name in unique_models:
        model_df = df[df["Model"] == model_name]
        y_true = model_df["y_true"].values
        y_pred = model_df["y_pred"].values

        # Compute clinical Net Benefit curve array
        nb_model = calculate_net_benefit(y_true, y_pred, thresholds)

        # Apply standardized color and title schemas
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8

        ax.plot(thresholds, nb_model, color=color, linewidth=linewidth, label=label)

    # Academic Formatting Adjustments 
    ax.set_xlim(0, 0.85)
    ax.set_ylim(ymin=-0.02, ymax=max(prevalence * 1.15, 0.25))

    ax.set_xlabel("Threshold Probability ($p_t$)", labelpad=10)
    ax.set_ylabel("Net Benefit", labelpad=10)
    # ax.set_title("Decision Curve Analysis", pad=15, fontweight="bold")

    # CRITICAL UPDATE: Shifted legend completely outside to the right of the grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))

    # Clean borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)

    # Save at 600 DPI; bbox_inches='tight' will auto-expand the canvas for the out-of-bounds legend
    plt.savefig("decision_curve_final_publication_all.pdf", format='pdf', bbox_inches="tight")
    plt.close(fig)
    print("Saved: decision_curve_final_publication.pdf")


# ==========================================
# 3. RUN EXECUTION
# ==========================================

local_csv_path = 'all_test_predictionsN.csv'
plot_decision_curve_reviewed(local_csv_path)

Reading variables from local path: all_test_predictionsN.csv
Plotting DCA for models: ['svm', 'xgb', 'bbn', 'nn', 'lr', 'rf', 'stack', 'sofa', 'apache']
Saved: decision_curve_final_publication.pdf


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

# ==========================================
# 1. JOURNAL-STYLE TYPOGRAPHY & SIZE CONFIG
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',          # Clean academic serif layout
    'font.size': 12,                # Baselines
    'axes.labelsize': 12,           # Feature axes labels
    'axes.titlesize': 12,           # Standalone titles
    'xtick.labelsize': 12,          # Ticks
    'ytick.labelsize': 12,          # Ticks
    'legend.fontsize': 12,          # Model legends
    'legend.frameon': False,        # Clean minimalist layout
    'savefig.dpi': 600,             # Print-ready high resolution
    'savefig.bbox': 'tight'
})

# Complete explicit aesthetic mapping (Colors optimized for maximum distinction)
MODEL_COLORS = {
    'stack':   '#d62728',  # Bold Crimson
    'xgb':     '#ff7f0e',  # Safety Orange
    'rf':      "#2c0c3a",  # Forest Green
    'svm':     '#1f77b4',  # Soft Blue
    'lr':      '#9467bd',  # Royal Purple
    'nn':      '#17becf',  # Deep Teal
    'bbn':     '#800040',  # Deep Plum Burgundy
    'sofa':    "#267c40",  # Amber Gold  ← was duplicate green
    'apache':  '#4a5568'   # Slate Charcoal
}

MODEL_LABELS = {
    'stack': 'STACKING',
    'xgb': 'XGBoost',
    'rf': 'Random Forest',
    'svm': 'SVM',
    'lr': 'Logistic Regression',
    'nn': 'MLP',
    'bbn': 'Bayesian Belief Network',
    'SOFA': 'SOFA Score',
    'APACHE': 'APACHE Score'
}

# Standardized to lowercase to guarantee correct string matching against normalization layers
MODELS_TO_PLOT = ['stack', 'xgb', 'lr', 'rf', 'bbn', 'nn', 'svm', 'SOFA', 'APACHE']

# ==========================================
# 2. LOW-RAM PLOTTING COMPILATION
# ==========================================
def generate_individual_plots(csv_path):
    print(f"Reading subset variables from local path: {csv_path}")
    
    # LOW-RAM OPTIMIZATION 1: Drop columns we do not need right at the disk-read layer
    needed_cols = ['Model', 'y_true', 'y_pred']
    
    # LOW-RAM OPTIMIZATION 2: Force data types to minimize memory footprint
    dtypes = {'Model': 'category', 'y_true': 'int8', 'y_pred': 'float32'}
    
    raw_df = pd.read_csv(csv_path, usecols=needed_cols, dtype=dtypes)
    
    # Standardize string structure safely
    raw_df['Model'] = raw_df['Model'].astype(str).str.lower().str.strip()
    
    # Filter using your model target array
    if MODELS_TO_PLOT is not None:
        target_models = [m.lower().strip() for m in MODELS_TO_PLOT]
        df = raw_df[raw_df['Model'].isin(target_models)].copy()
    else:
        df = raw_df
        
    unique_models = df['Model'].unique()
    print(f"Plotting active subset configurations: {list(unique_models)}")

    fig_size = (7.5, 6.5)

    # -------------------------------------------------------------------------
    # FIGURE 1: ROC Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Chance')
    
    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        fpr, tpr, _ = roc_curve(m_df['y_true'], m_df['y_pred'])
        roc_auc = auc(fpr, tpr)
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(fpr, tpr, color=color, lw=linewidth,
                label=f'{label} (AUC = {roc_auc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', labelpad=10)
    ax.set_ylabel('True Positive Rate (Sensitivity)', labelpad=10)
    # ax.set_title('Receiver Operating Characteristic (ROC)', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_roc_curve.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_roc_curve.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 2: Precision-Recall Curve (AUPRC)
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    baseline = df['y_true'].sum() / len(df['y_true'])
    ax.plot([0, 1], [baseline, baseline], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label=f'Baseline ({baseline:.2f})')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        precision, recall, _ = precision_recall_curve(m_df['y_true'], m_df['y_pred'])
        auprc = average_precision_score(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(recall, precision, color=color, lw=linewidth,
                label=f'{label} (AUPRC = {auprc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Recall (Sensitivity)', labelpad=10)
    ax.set_ylabel('Precision (Positive Predictive Value)', labelpad=10)
    # ax.set_title('Precision-Recall Curve (PRC)', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_precision_recall_curve.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_precision_recall_curve.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 3: Calibration Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Perfect Calibration')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        prob_true, prob_pred = calibration_curve(m_df['y_true'], m_df['y_pred'], n_bins=10, strategy='uniform')
        brier = brier_score_loss(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.0 if model_name == 'stacking' else 1.4
        marker_size = 5 if model_name == 'stacking' else 4
        
        ax.plot(prob_pred, prob_true, marker='o', markersize=marker_size, color=color, lw=linewidth,
                label=f'{label} (Brier = {brier:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Mean Predicted Probability', labelpad=10)
    ax.set_ylabel('Fraction of Positives', labelpad=10)
    # ax.set_title('Calibration Curve', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_calibration_curve.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_calibration_curve.pdf")

# ==========================================
# 3. RUN EXECUTION FROM LOCAL DIRECTORY
# ==========================================
if __name__ == "__main__":
    local_csv_path = 'all_test_predictionsN.csv' 
    generate_individual_plots(local_csv_path)

Reading subset variables from local path: all_test_predictionsN.csv
Plotting active subset configurations: ['svm', 'xgb', 'bbn', 'nn', 'lr', 'rf', 'stack', 'sofa', 'apache']
Saved: fig_roc_curve.pdf
Saved: fig_precision_recall_curve.pdf
Saved: fig_calibration_curve.pdf


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

# ==========================================
# 1. JOURNAL-STYLE TYPOGRAPHY & SIZE CONFIG
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',          # Clean academic serif layout
    'font.size': 12,                # Baselines
    'axes.labelsize': 12,           # Feature axes labels
    'axes.titlesize': 12,           # Standalone titles
    'xtick.labelsize': 12,          # Ticks
    'ytick.labelsize': 12,          # Ticks
    'legend.fontsize': 12,          # Model legends
    'legend.frameon': False,        # Clean minimalist layout
    'savefig.dpi': 600,             # Print-ready high resolution
    'savefig.bbox': 'tight'
})

# Complete explicit aesthetic mapping (Colors optimized for maximum distinction)
MODEL_COLORS = {
    'stack': '#d62728',  # Bold Crimson
    'xgb': '#ff7f0e',       # Safety Orange
    'rf': '#2ca02c',        # Forest Green
    'svm': '#1f77b4',       # Soft Blue
    'lr': '#9467bd',        # Royal Purple
    'nn': '#17becf',        # Deep Teal
    'bbn': '#800040',       # DISTINCT: Deep Plum Burgundy
    'sofa': "#0b320b",      # Matching palette consistency
    'apache': '#4a5568'     # DISTINCT: Slate Charcoal/Navy
}

MODEL_LABELS = {
    'stack': 'STACKING',
    'xgb': 'XGBoost',
    'rf': 'Random Forest',
    'svm': 'SVM',
    'lr': 'Logistic Regression',
    'mlp': 'MLP',
    'bbn': 'Bayesian Belief Network',
    'SOFA': 'SOFA Score',
    'APACHE': 'APACHE Score'
}

# Standardized to lowercase to guarantee correct string matching against normalization layers
MODELS_TO_PLOT = ['stack', 'xgb', 'lr', 'SOFA', 'APACHE']

# ==========================================
# 2. LOW-RAM PLOTTING COMPILATION
# ==========================================
def generate_individual_plots(csv_path):
    print(f"Reading subset variables from local path: {csv_path}")
    
    # LOW-RAM OPTIMIZATION 1: Drop columns we do not need right at the disk-read layer
    needed_cols = ['Model', 'y_true', 'y_pred']
    
    # LOW-RAM OPTIMIZATION 2: Force data types to minimize memory footprint
    dtypes = {'Model': 'category', 'y_true': 'int8', 'y_pred': 'float32'}
    
    raw_df = pd.read_csv(csv_path, usecols=needed_cols, dtype=dtypes)
    
    # Standardize string structure safely
    raw_df['Model'] = raw_df['Model'].astype(str).str.lower().str.strip()
    
    # Filter using your model target array
    if MODELS_TO_PLOT is not None:
        target_models = [m.lower().strip() for m in MODELS_TO_PLOT]
        df = raw_df[raw_df['Model'].isin(target_models)].copy()
    else:
        df = raw_df
        
    unique_models = df['Model'].unique()
    print(f"Plotting active subset configurations: {list(unique_models)}")

    fig_size = (7.5, 6.5)

    # -------------------------------------------------------------------------
    # FIGURE 1: ROC Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Chance')
    
    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        fpr, tpr, _ = roc_curve(m_df['y_true'], m_df['y_pred'])
        roc_auc = auc(fpr, tpr)
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(fpr, tpr, color=color, lw=linewidth,
                label=f'{label} (AUC = {roc_auc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', labelpad=10)
    ax.set_ylabel('True Positive Rate (Sensitivity)', labelpad=10)
    # ax.set_title('Receiver Operating Characteristic (ROC)', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_roc_curve_s.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_roc_curve.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 2: Precision-Recall Curve (AUPRC)
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    baseline = df['y_true'].sum() / len(df['y_true'])
    ax.plot([0, 1], [baseline, baseline], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label=f'Baseline ({baseline:.2f})')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        precision, recall, _ = precision_recall_curve(m_df['y_true'], m_df['y_pred'])
        auprc = average_precision_score(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(recall, precision, color=color, lw=linewidth,
                label=f'{label} (AUPRC = {auprc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Recall (Sensitivity)', labelpad=10)
    ax.set_ylabel('Precision (Positive Predictive Value)', labelpad=10)
    # ax.set_title('Precision-Recall Curve (PRC)', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_precision_recall_curve_s.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_precision_recall_curve.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 3: Calibration Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Perfect Calibration')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        prob_true, prob_pred = calibration_curve(m_df['y_true'], m_df['y_pred'], n_bins=10, strategy='uniform')
        brier = brier_score_loss(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        label = MODEL_LABELS.get(model_name, model_name.upper())
        linewidth = 2.0 if model_name == 'stacking' else 1.4
        marker_size = 5 if model_name == 'stacking' else 4
        
        ax.plot(prob_pred, prob_true, marker='o', markersize=marker_size, color=color, lw=linewidth,
                label=f'{label} (Brier = {brier:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Mean Predicted Probability', labelpad=10)
    ax.set_ylabel('Fraction of Positives', labelpad=10)
    # ax.set_title('Calibration Curve', weight='bold', pad=15)
    
    # SHIFTED OUTSIDE: Legend positioned perfectly to the right side of the box grid
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_calibration_curve_s.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)
    print("Saved: fig_calibration_curve.pdf")

# ==========================================
# 3. RUN EXECUTION FROM LOCAL DIRECTORY
# ==========================================
if __name__ == "__main__":
    local_csv_path = 'all_test_predictionsN.csv' 
    generate_individual_plots(local_csv_path)

Reading subset variables from local path: all_test_predictionsN.csv
Plotting active subset configurations: ['xgb', 'lr', 'stack', 'sofa', 'apache']
Saved: fig_roc_curve.pdf
Saved: fig_precision_recall_curve.pdf
Saved: fig_calibration_curve.pdf
